# CUT ML — cleaned Scene_change_detector notebook

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import json
import zipfile
import shutil
import re
from pathlib import Path

# При необходимости просто измени путь ниже на свой zip-архив на Google Drive.
ZIP_PATH = Path("/content/drive/MyDrive/public_tests.zip")

RAW_DIR = Path("/content/_raw_dataset")
DATASET_DIR = Path("/content/train_dataset")

assert ZIP_PATH.exists(), f"Не найден архив: {ZIP_PATH}"

if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

RAW_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(RAW_DIR)

def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f, strict=False)

ready_train_dirs = [p.parent for p in RAW_DIR.rglob("info.json") if p.parent.name == "train_dataset"]

if ready_train_dirs:
    shutil.copytree(ready_train_dirs[0], DATASET_DIR)
    print("✅ Найден готовый train_dataset внутри архива")
else:
    case_dirs = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_dir() and re.match(r"^\d+_test_file_input$", p.name)
    )
    assert case_dirs, "Не найден ни train_dataset, ни папки вида *_test_file_input"

    (DATASET_DIR / "video").mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / "gt").mkdir(parents=True, exist_ok=True)

    def resolve_video_and_gt(case_input_dir: Path, case_id: str):
        info_path = case_input_dir / "info.json"
        assert info_path.exists(), f"Нет info.json в {case_input_dir}"

        info = load_json(info_path)
        if isinstance(info, dict):
            info = [info]
        assert len(info) > 0, f"Пустой info.json в {case_input_dir}"
        rec = info[0]

        src_rel = rec.get("source")
        gt_rel = rec.get("scene_change") or rec.get("gt")

        video_path = (case_input_dir / src_rel) if src_rel else None
        if video_path is None or not video_path.exists():
            videos = sorted((case_input_dir / "video").glob("*.mp4"))
            assert videos, f"Не найден mp4 в {case_input_dir / 'video'}"
            video_path = videos[0]

        gt_path = (case_input_dir / gt_rel) if gt_rel else None
        if gt_path is None or not gt_path.exists():
            gts = sorted((case_input_dir / "gt").glob("*.json"))
            if gts:
                gt_path = gts[0]
            else:
                sibling_gt = case_input_dir.parent / f"{case_id}_test_file_gt"
                assert sibling_gt.exists(), f"Не найден {sibling_gt}"
                gts2 = sorted(sibling_gt.rglob("*.json"))
                assert gts2, f"Есть {sibling_gt}, но json не найден"
                gt_path = gts2[0]

        return video_path, gt_path, rec

    merged_info = []
    for case_dir in case_dirs:
        case_id = case_dir.name.split("_")[0]
        video_src, gt_src, rec = resolve_video_and_gt(case_dir, case_id)

        video_dst = DATASET_DIR / "video" / f"{case_id}.mp4"
        gt_dst = DATASET_DIR / "gt" / f"{case_id}.json"

        shutil.copy2(video_src, video_dst)
        shutil.copy2(gt_src, gt_dst)

        new_rec = dict(rec)
        new_rec["source"] = f"video/{case_id}.mp4"
        new_rec["scene_change"] = f"gt/{case_id}.json"
        merged_info.append(new_rec)

    with open(DATASET_DIR / "info.json", "w", encoding="utf-8") as f:
        json.dump(merged_info, f, ensure_ascii=False, indent=2)

    print("✅ train_dataset собран из *_test_file_input")

print("Папка датасета:", DATASET_DIR)
print("Видео:", len(list((DATASET_DIR / "video").glob("*.mp4"))))
print("GT:", len(list((DATASET_DIR / "gt").glob("*.json"))))

os.chdir("/content")
print("Текущая папка:", os.getcwd())

Mounted at /content/drive
✅ train_dataset собран из *_test_file_input
Папка датасета: /content/train_dataset
Видео: 10
GT: 10
Текущая папка: /content


In [ ]:
pip install opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 57.1 MB/s eta 0:00:00


In [ ]:
import os
import json
import pickle
import numpy as np
import cv2

from pathlib import Path
from tqdm.auto import tqdm
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score as sklearn_f1_score

In [ ]:
def load_json_from_file(filename):
    with open(filename, "r") as f:
        return json.load(f, strict=False)

def dump_json_to_file(obj, filename, **kwargs):
    with open(filename, "w") as f:
        json.dump(obj, f, **kwargs)

def read_video(video_path):
    cap = cv2.VideoCapture(video_path)
    while cap.isOpened():
        ret, frame = cap.read()
        if ret is False:
            break
        yield frame
    cap.release()

def get_video_len(video_path):
    cap = cv2.VideoCapture(video_path)
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return length

def calculate_matrix(true_scd, predicted_scd, scene_len, not_to_use_frames=set()):
    predicted_scd = set(predicted_scd)
    true_scd = set(true_scd)
    tp, fp, tn, fn = 0, 0, 0, 0
    for scd in predicted_scd:
        if scd in true_scd:
            tp += 1
        elif scd not in not_to_use_frames:
            fp += 1
    for scd in true_scd:
        if scd not in predicted_scd:
            fn += 1
    tn = scene_len - len(not_to_use_frames) - tp - fp - fn
    return tp, fp, tn, fn

def calculate_precision(tp, fp, tn, fn):
    return tp / max(1, (tp + fp))

def calculate_recall(tp, fp, tn, fn):
    return tp / max(1, (tp + fn))

def f1_score(true_scd, predicted_scd, scene_len, not_to_use_frames=set()):
    tp, fp, tn, fn = calculate_matrix(true_scd, predicted_scd, scene_len, not_to_use_frames)
    precision_score = calculate_precision(tp, fp, tn, fn)
    recall_score = calculate_recall(tp, fp, tn, fn)
    if precision_score + recall_score == 0:
        return 0
    return 2 * precision_score * recall_score / (precision_score + recall_score)

def f1_score_matrix(tp, fp, tn, fn):
    precision_score = calculate_precision(tp, fp, tn, fn)
    recall_score = calculate_recall(tp, fp, tn, fn)
    if precision_score + recall_score == 0:
        return 0
    return 2 * precision_score * recall_score / (precision_score + recall_score)

In [ ]:
def baseline_scene_change_detector(frames, threshold=2000, with_vis=False):
    def pixel_metric(frame, prev_frame):
        return np.mean((frame.astype(np.int32) - prev_frame) ** 2)

    scene_changes = []
    vis = []
    metric_values = []
    prev_frame = None

    for idx, frame in enumerate(frames):
        if prev_frame is not None:
            metric_value = pixel_metric(frame, prev_frame)
            if metric_value > threshold:
                scene_changes.append(idx)
                if with_vis and len(vis) < 100:
                    vis.append([prev_frame, frame])
            metric_values.append(metric_value)
        else:
            metric_values.append(0)
        prev_frame = frame

    return scene_changes, vis, metric_values

In [ ]:
video_info_list = load_json_from_file(DATASET_DIR / "info.json")
video_ids = [Path(item["source"]).stem for item in video_info_list]

HOLDOUT_VIDEO_ID = "08" if "08" in video_ids else video_ids[-1]
train_videos = [video_id for video_id in video_ids if video_id != HOLDOUT_VIDEO_ID]

print("Все видео:", video_ids)
print("Holdout video:", HOLDOUT_VIDEO_ID)
print("Train videos:", train_videos)

Все видео: ['00', '01', '02', '03', '04', '05', '06', '07', '08', '09']
Holdout video: 08
Train videos: ['00', '01', '02', '03', '04', '05', '06', '07', '09']


In [ ]:
def get_train_data(train_videos, dataset_path):
    X_train, y_train = np.array([]), np.array([])
    dataset_path = str(dataset_path)

    for video in train_videos:
        frames = read_video(os.path.join(dataset_path, "video", f"{video}.mp4"))
        _, _, metric_values = baseline_scene_change_detector(frames)

        cuts = load_json_from_file(os.path.join(dataset_path, "gt", f"{video}.json")).get("cut", [])
        video_scenes = np.zeros(len(metric_values), dtype=np.uint8)

        valid_cuts = [cut for cut in cuts if 0 <= cut < len(video_scenes)]
        if valid_cuts:
            video_scenes[np.array(valid_cuts, dtype=np.int32)] = 1

        X_train = np.hstack((X_train, np.array(metric_values, dtype=np.float32)))
        y_train = np.hstack((y_train, video_scenes))

    return X_train, y_train

In [ ]:
X_train, y_train = get_train_data(train_videos, DATASET_DIR)

params = {"kernel": "rbf", "C": 1}
svm_model = SVC(**params)
svm_model.fit(X_train.reshape(-1, 1), y_train)

with open("model.pkl", "wb") as f:
    pickle.dump(svm_model, f)

print("SVM baseline trained and saved to model.pkl")

SVM baseline trained and saved to model.pkl


In [ ]:
def baseline_scene_change_detection_ml(frames, threshold=0.5):
    _, _, metric_values = baseline_scene_change_detector(frames)
    X_test = np.array(metric_values, dtype=np.float32).reshape(-1, 1)

    model = pickle.load(open("model.pkl", "rb"))
    predict_cuts = model.predict(X_test)

    return np.where(predict_cuts > 0)[0], [], metric_values

In [ ]:
def run_scene_change_detector_ml_one_video(scene_change_detector, dataset_path, video_num):
    video_meta = load_json_from_file(os.path.join(dataset_path, "info.json"))[video_num]
    video_path = os.path.join(dataset_path, video_meta["source"])
    frames = read_video(video_path)
    video_len = video_meta.get("len", get_video_len(video_path))
    true_scene_changes = load_json_from_file(os.path.join(dataset_path, video_meta["scene_change"]))

    not_use_frames = set()
    for type_scene_change in ["trash", "fade", "dissolve"]:
        for bad_scene_range in true_scene_changes.get(type_scene_change, []):
            not_use_frames.update(range(bad_scene_range[0], bad_scene_range[1] + 1))

    predicted_scene_changes, _, _ = scene_change_detector(frames)

    if isinstance(predicted_scene_changes, np.ndarray):
        predicted_scene_changes = predicted_scene_changes.tolist()

    true_cuts = true_scene_changes.get("cut", [])
    return f1_score(true_cuts, predicted_scene_changes, video_len, not_use_frames)

def run_scene_change_detector_all_video(scene_change_detector, dataset_path):
    dataset_path = str(dataset_path)
    video_dataset = load_json_from_file(os.path.join(dataset_path, "info.json"))
    param_log = {"_mean_f1_score": []}

    for video_meta in tqdm(video_dataset, total=len(video_dataset), desc="Processing videos"):
        video_path = os.path.join(dataset_path, video_meta["source"])
        frames = read_video(video_path)
        video_len = video_meta.get("len", get_video_len(video_path))
        true_scene_changes = load_json_from_file(os.path.join(dataset_path, video_meta["scene_change"]))

        not_use_frames = set()
        for type_scene_change in ["trash", "fade", "dissolve"]:
            for bad_scene_range in true_scene_changes.get(type_scene_change, []):
                not_use_frames.update(range(bad_scene_range[0], bad_scene_range[1] + 1))

        predicted_scene_changes, _, _ = scene_change_detector(frames)

        param_log[f"f1_score_{video_meta['source']}"] = f1_score(
            true_scene_changes.get("cut", []),
            predicted_scene_changes,
            video_len,
            not_use_frames,
        )
        video_tp, video_fp, video_tn, video_fn = calculate_matrix(
            true_scene_changes.get("cut", []),
            predicted_scene_changes,
            video_len,
            not_use_frames,
        )

        param_log[f"tp_{video_meta['source']}"] = video_tp
        param_log[f"fp_{video_meta['source']}"] = video_fp
        param_log[f"tn_{video_meta['source']}"] = video_tn
        param_log[f"fn_{video_meta['source']}"] = video_fn
        param_log["_mean_f1_score"].append(param_log[f"f1_score_{video_meta['source']}"])

    param_log["_mean_f1_score"] = float(np.mean(param_log["_mean_f1_score"])) if param_log["_mean_f1_score"] else 0.0
    return param_log

In [ ]:
# Variant 23 - HistGB Shallow Regularized (minimal safe port from ML_CUT_0_951.ipynb)
# Train videos are expected in `train_videos`; optional `test_videos` / `holdout_videos` may be present for local evaluation.

import os
import gc
import json
import pickle
import argparse
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

DATASET_DIR = os.environ.get('DATASET_DIR', 'train_dataset')

# -----------------------------------------------------------------------------
# Exact experiment config for exp_id=23 HistGB Shallow Regularized
# -----------------------------------------------------------------------------
RESIZE_HW = (96, 54)
HIST_BINS = 32
DIFF_THRESHOLD = 11
BLOCK_GRIDS = (4, 8, 12)
BLOCK_MEAN_THRESHOLD = 12.0
CENTER_CROP_FRAC = 0.5

EXP19_CANDIDATE_CFG = {"mode": "base_peaks", "q": 0.75, "radius": 1}
EXP19_FEATURE_CFG = {
    "wider_windows": False,
    "second_order": False,
    "local_rank": False,
    "cross_ratios": False,
    "agreement": False,
    "asymmetry": False,
    "peak_shape": False,
    "local_energy": False,
    "stability": False,
    "include_ycbcr": False,
    "include_sobel": False,
    "include_multigrid": True,
    "include_center_border": False,
    "include_teacher": False,
}
EXP19_MODEL_CFG = {
    "kind": "histgb",
    "learning_rate": 0.05,
    "max_depth": 3,
    "max_iter": 260,
    "min_samples_leaf": 30,
    "l2_regularization": 0.03,
    "random_state": 42,
    "sample_weight_mode": "none",
}
EXP19_THR = 0.1000
EXP19_DIST = 4

BASE_SIGNAL_KEYS = [
    "mafd_norm", "diff_ratio", "hsv", "chrom", "hist_l1", "edge",
    "mean_delta", "std_delta", "sat_delta",
    "block_mean", "block_max", "block_ratio",
    "mafd_l2", "diff_ratio_l2", "hist_l1_l2", "block_mean_l2",
    "mafd_l3", "diff_ratio_l3",
    "heuristic_fusion", "heuristic_robust", "heuristic_multiscale",
]
MULTIGRID_KEYS = [
    "block_mean_g4", "block_max_g4", "block_ratio_g4",
    "block_mean_g12", "block_max_g12", "block_ratio_g12",
]

# -----------------------------------------------------------------------------
# Utilities
# -----------------------------------------------------------------------------
def _load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f, strict=False)


def load_json_from_file(path):
    return _load_json(path)


def shift_array(x, offset):
    x = np.asarray(x, dtype=np.float32)
    if x.size == 0 or offset == 0:
        return x.copy()
    out = np.empty_like(x)
    if offset > 0:
        out[:offset] = x[0]
        out[offset:] = x[:-offset]
    else:
        off = -offset
        out[-off:] = x[-1]
        out[:-off] = x[off:]
    return out.astype(np.float32)


def rolling_mean_1d(x, window):
    x = np.asarray(x, dtype=np.float32)
    if x.size == 0:
        return x.copy()
    if window <= 1:
        return x.copy()
    pad = window // 2
    padded = np.pad(x, (pad, pad), mode='edge')
    kernel = np.ones(window, dtype=np.float32) / float(window)
    return np.convolve(padded, kernel, mode='valid').astype(np.float32)


def rolling_std_1d(x, window):
    x = np.asarray(x, dtype=np.float32)
    mean = rolling_mean_1d(x, window)
    mean_sq = rolling_mean_1d(x * x, window)
    var = np.maximum(0.0, mean_sq - mean * mean)
    return np.sqrt(var).astype(np.float32)


def local_peak_1d(x, window=5):
    return (np.asarray(x, dtype=np.float32) - rolling_mean_1d(x, window)).astype(np.float32)


def local_ratio_1d(x, window=5):
    x = np.asarray(x, dtype=np.float32)
    return (x / (rolling_mean_1d(x, window) + 1e-6)).astype(np.float32)


def robust_zscore(x):
    x = np.asarray(x, dtype=np.float32)
    if x.size == 0:
        return x.copy()
    med = float(np.median(x))
    mad = float(np.median(np.abs(x - med)))
    z = (x - med) / (1.4826 * mad + 1e-6)
    return np.clip(z, -10.0, 10.0).astype(np.float32)


def percentile_scale(x, low=5.0, high=95.0):
    x = np.asarray(x, dtype=np.float32)
    if x.size == 0:
        return x.copy()
    lo, hi = np.percentile(x, [low, high])
    scaled = (x - lo) / (hi - lo + 1e-6)
    return np.clip(scaled, 0.0, 1.0).astype(np.float32)


def local_peak_mask(scores):
    scores = np.asarray(scores, dtype=np.float32)
    if scores.size == 0:
        return np.zeros(0, dtype=bool)
    left = np.empty_like(scores)
    right = np.empty_like(scores)
    left[0] = scores[0]
    left[1:] = scores[:-1]
    right[-1] = scores[-1]
    right[:-1] = scores[1:]
    return (scores >= left) & (scores > right)


def expand_mask(mask, radius=1):
    mask = np.asarray(mask, dtype=bool)
    if radius <= 0 or mask.size == 0:
        return mask
    idx = np.where(mask)[0]
    out = mask.copy()
    for i in idx:
        left = max(0, i - radius)
        right = min(mask.size, i + radius + 1)
        out[left:right] = True
    return out


def calculate_matrix(true_scd, predicted_scd, scene_len, not_to_use_frames=set()):
    predicted_scd = set(int(x) for x in predicted_scd)
    true_scd = set(int(x) for x in true_scd)
    tp, fp, tn, fn = 0, 0, 0, 0
    for scd in predicted_scd:
        if scd in true_scd:
            tp += 1
        elif scd not in not_to_use_frames:
            fp += 1
    for scd in true_scd:
        if scd not in predicted_scd:
            fn += 1
    tn = scene_len - len(not_to_use_frames) - tp - fp - fn
    return tp, fp, tn, fn


def calculate_precision(tp, fp, tn, fn):
    return tp / max(tp + fp, 1)


def calculate_recall(tp, fp, tn, fn):
    return tp / max(tp + fn, 1)


def calculate_f1_score(tp, fp, tn, fn):
    precision = calculate_precision(tp, fp, tn, fn)
    recall = calculate_recall(tp, fp, tn, fn)
    if precision + recall == 0:
        return 0.0
    return 2.0 * precision * recall / (precision + recall)


def evaluate_predictions(entry, predicted_cuts):
    tp, fp, tn, fn = calculate_matrix(
        true_scd=entry['cuts'],
        predicted_scd=predicted_cuts,
        scene_len=entry['video_len'],
        not_to_use_frames=set(np.where(entry['ignore_mask'])[0].tolist()),
    )
    return {
        'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
        'precision': calculate_precision(tp, fp, tn, fn),
        'recall': calculate_recall(tp, fp, tn, fn),
        'f1': calculate_f1_score(tp, fp, tn, fn),
        'pred_count': len(predicted_cuts),
    }


def greedy_distance_filter(indices, scores, min_distance):
    indices = np.asarray(indices, dtype=np.int32)
    if indices.size == 0:
        return []
    order = indices[np.argsort(scores[indices])[::-1]]
    keep = []
    for idx in order:
        if all(abs(int(idx) - int(prev)) >= int(min_distance) for prev in keep):
            keep.append(int(idx))
    return sorted(keep)


def postprocess_fixed(scores, threshold, min_distance=1, peaks_only=True):
    scores = np.asarray(scores, dtype=np.float32)
    if scores.size == 0:
        return []
    mask = scores >= float(threshold)
    if peaks_only:
        mask = mask & local_peak_mask(scores)
    return greedy_distance_filter(np.where(mask)[0], scores, min_distance=min_distance)


def read_video_to_list(video_path):
    frames = []
    cap = cv2.VideoCapture(str(video_path))
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    return frames


def make_hist(gray, hist_bins=32):
    hist = cv2.calcHist([gray], [0], None, [hist_bins], [0, 256]).astype(np.float32).reshape(-1)
    hist /= max(hist.sum(), 1.0)
    return hist


def make_hist_flat(values, hist_bins=32):
    values = np.asarray(values, dtype=np.uint8).reshape(-1, 1)
    hist = cv2.calcHist([values], [0], None, [hist_bins], [0, 256]).astype(np.float32).reshape(-1)
    hist /= max(hist.sum(), 1.0)
    return hist


def compute_block_means(gray, grid):
    h, w = gray.shape
    block_h = max(1, h // grid)
    block_w = max(1, w // grid)
    out = []
    for gy in range(grid):
        for gx in range(grid):
            y0 = gy * block_h
            y1 = h if gy == grid - 1 else (gy + 1) * block_h
            x0 = gx * block_w
            x1 = w if gx == grid - 1 else (gx + 1) * block_w
            out.append(float(gray[y0:y1, x0:x1].mean()))
    return np.asarray(out, dtype=np.float32)


def align_lag_signal(values, n_transitions, lag):
    values = np.asarray(values, dtype=np.float32)
    out = np.zeros(n_transitions, dtype=np.float32)
    if values.size == 0 or n_transitions == 0:
        return out
    start = lag - 1
    end = min(n_transitions, start + values.size)
    out[start:end] = values[:max(0, end - start)]
    if start > 0:
        out[:start] = out[start]
    if end < n_transitions and end > 0:
        out[end:] = out[end - 1]
    return out


def compute_heuristic_scores(base_signals):
    diff_ratio = percentile_scale(base_signals['diff_ratio'])
    hsv = percentile_scale(base_signals['hsv'])
    chrom = percentile_scale(base_signals['chrom'])
    mafd = percentile_scale(base_signals['mafd_norm'])
    block_ratio = percentile_scale(base_signals['block_ratio'])
    hist = percentile_scale(base_signals['hist_l1'])
    edge = percentile_scale(base_signals['edge'])
    sobel = percentile_scale(base_signals.get('sobel_diff', base_signals['edge']))

    heuristic_fusion = (
        0.28 * diff_ratio + 0.24 * hsv + 0.28 * chrom + 0.10 * mafd +
        0.05 * hist + 0.03 * edge + 0.02 * block_ratio
    ).astype(np.float32)
    heuristic_fusion = percentile_scale(heuristic_fusion)

    z_diff = np.clip(robust_zscore(base_signals['diff_ratio']), 0.0, None)
    z_hist = np.clip(robust_zscore(base_signals['hist_l1']), 0.0, None)
    z_block = np.clip(robust_zscore(base_signals['block_ratio']), 0.0, None)
    z_mafd = np.clip(robust_zscore(base_signals['mafd_norm']), 0.0, None)
    z_chrom = np.clip(robust_zscore(base_signals['chrom']), 0.0, None)
    z_edge = np.clip(robust_zscore(base_signals['edge']), 0.0, None)
    z_sobel = np.clip(robust_zscore(sobel), 0.0, None)

    lp_diff = np.clip(local_peak_1d(base_signals['diff_ratio'], 5), 0.0, None)
    lp_mafd = np.clip(local_peak_1d(base_signals['mafd_norm'], 5), 0.0, None)
    lp_block = np.clip(local_peak_1d(base_signals['block_ratio'], 5), 0.0, None)

    heuristic_robust = (
        0.18 * percentile_scale(z_diff) + 0.12 * percentile_scale(z_hist) +
        0.12 * percentile_scale(z_block) + 0.12 * percentile_scale(z_mafd) +
        0.10 * percentile_scale(z_chrom) + 0.08 * percentile_scale(z_edge) +
        0.06 * percentile_scale(z_sobel) + 0.08 * percentile_scale(lp_diff) +
        0.08 * percentile_scale(lp_mafd) + 0.06 * percentile_scale(lp_block)
    ).astype(np.float32)
    heuristic_robust = percentile_scale(heuristic_robust)

    heuristic_multiscale = (
        0.24 * percentile_scale(base_signals['mafd_norm']) +
        0.16 * percentile_scale(base_signals['diff_ratio']) +
        0.12 * percentile_scale(base_signals['hist_l1']) +
        0.08 * percentile_scale(base_signals['block_ratio']) +
        0.10 * percentile_scale(base_signals['mafd_l2']) +
        0.08 * percentile_scale(base_signals['diff_ratio_l2']) +
        0.05 * percentile_scale(base_signals['hist_l1_l2']) +
        0.03 * percentile_scale(base_signals['block_mean_l2']) +
        0.08 * percentile_scale(base_signals['mafd_l3']) +
        0.06 * percentile_scale(base_signals['diff_ratio_l3'])
    ).astype(np.float32)
    heuristic_multiscale = percentile_scale(heuristic_multiscale)

    heuristic_teacher = percentile_scale(0.45 * heuristic_robust + 0.35 * heuristic_multiscale + 0.20 * heuristic_fusion)

    return {
        'heuristic_fusion': heuristic_fusion,
        'heuristic_robust': heuristic_robust,
        'heuristic_multiscale': heuristic_multiscale,
        'heuristic_teacher': heuristic_teacher,
    }


def extract_video_cache_from_frames(frames, video_id, gt):
    width, height = RESIZE_HW
    gray_frames = []
    hist_frames = []
    block3_frames = []

    mae_list = []
    mse_list = []
    diff_ratio_list = []
    hsv_list = []
    chrom_list = []
    hist_list = []
    edge_list = []
    sobel_diff_list = []
    mean_delta_list = []
    std_delta_list = []
    sat_delta_list = []
    y_delta_list = []
    cr_delta_list = []
    cb_delta_list = []
    block_mean_lists = {g: [] for g in BLOCK_GRIDS}
    block_max_lists = {g: [] for g in BLOCK_GRIDS}
    block_ratio_lists = {g: [] for g in BLOCK_GRIDS}
    mafd_norm_list = []
    center_diff_ratio_list = []
    border_diff_ratio_list = []
    center_hist_l1_list = []
    border_hist_l1_list = []

    prev_gray = None
    prev_hsv = None
    prev_ycrcb = None
    prev_hist = None
    prev_lap = None
    prev_sobel = None
    prev_mean = None
    prev_std = None
    prev_sat = None
    prev_blocks = {g: None for g in BLOCK_GRIDS}
    prev_center_gray = None
    prev_border_gray = None
    prev_center_hist = None
    prev_border_hist = None

    for frame in frames:
        if frame is None:
            continue
        small = cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA)
        gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
        hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)
        ycrcb = cv2.cvtColor(small, cv2.COLOR_BGR2YCrCb)
        hist = make_hist(gray, hist_bins=HIST_BINS)
        lap = cv2.Laplacian(gray, cv2.CV_16S, ksize=3)
        sobelx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
        sobely = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
        sobel = cv2.magnitude(sobelx, sobely).astype(np.float32)

        cur_mean = float(gray.mean())
        cur_std = float(gray.std())
        cur_sat = float(hsv[:, :, 1].mean())
        block3 = cv2.resize(gray, (3, 3), interpolation=cv2.INTER_AREA).astype(np.float32).reshape(-1)
        blocks = {g: compute_block_means(gray, grid=g) for g in BLOCK_GRIDS}

        gray_frames.append(gray)
        hist_frames.append(hist)
        block3_frames.append(block3)

        h, w = gray.shape
        ch = max(1, int(round(h * CENTER_CROP_FRAC)))
        cw = max(1, int(round(w * CENTER_CROP_FRAC)))
        y0 = max(0, (h - ch) // 2)
        x0 = max(0, (w - cw) // 2)
        y1 = min(h, y0 + ch)
        x1 = min(w, x0 + cw)

        center_gray = gray[y0:y1, x0:x1]
        border_mask = np.ones_like(gray, dtype=bool)
        border_mask[y0:y1, x0:x1] = False
        border_gray = gray[border_mask]
        center_hist = make_hist(center_gray, hist_bins=HIST_BINS)
        border_hist = make_hist_flat(border_gray, hist_bins=HIST_BINS)

        if prev_gray is not None:
            diff = gray.astype(np.int16) - prev_gray.astype(np.int16)
            diff_f = diff.astype(np.float32)
            abs_diff = np.abs(diff_f)

            mae = float(abs_diff.mean())
            mse = float(np.mean(diff_f ** 2))
            mafd = float(mae / 255.0)
            diff_ratio = float(np.mean(abs_diff > DIFF_THRESHOLD))
            hsv_diff = float(np.mean(np.abs(hsv.astype(np.int16) - prev_hsv.astype(np.int16))) / 255.0)
            chrom_diff = float(np.mean(np.abs(ycrcb[:, :, 1:].astype(np.int16) - prev_ycrcb[:, :, 1:].astype(np.int16))) / 255.0)
            hist_l1 = float(np.abs(hist - prev_hist).sum())
            edge = float(np.mean(np.abs(lap.astype(np.int16) - prev_lap.astype(np.int16))) / 255.0)
            sobel_diff = float(np.mean(np.abs(sobel - prev_sobel)) / 255.0)

            y_delta = float(np.mean(np.abs(ycrcb[:, :, 0].astype(np.int16) - prev_ycrcb[:, :, 0].astype(np.int16))) / 255.0)
            cr_delta = float(np.mean(np.abs(ycrcb[:, :, 1].astype(np.int16) - prev_ycrcb[:, :, 1].astype(np.int16))) / 255.0)
            cb_delta = float(np.mean(np.abs(ycrcb[:, :, 2].astype(np.int16) - prev_ycrcb[:, :, 2].astype(np.int16))) / 255.0)

            mae_list.append(mae)
            mse_list.append(mse)
            diff_ratio_list.append(diff_ratio)
            hsv_list.append(hsv_diff)
            chrom_list.append(chrom_diff)
            hist_list.append(hist_l1)
            edge_list.append(edge)
            sobel_diff_list.append(sobel_diff)
            mean_delta_list.append(abs(cur_mean - prev_mean))
            std_delta_list.append(abs(cur_std - prev_std))
            sat_delta_list.append(abs(cur_sat - prev_sat))
            y_delta_list.append(y_delta)
            cr_delta_list.append(cr_delta)
            cb_delta_list.append(cb_delta)
            mafd_norm_list.append(mafd)

            for g in BLOCK_GRIDS:
                block_abs_g = np.abs(blocks[g] - prev_blocks[g])
                block_mean_lists[g].append(float(block_abs_g.mean()))
                block_max_lists[g].append(float(block_abs_g.max()))
                block_ratio_lists[g].append(float(np.mean(block_abs_g > BLOCK_MEAN_THRESHOLD)))

            center_diff = np.abs(center_gray.astype(np.int16) - prev_center_gray.astype(np.int16))
            border_diff = np.abs(border_gray.astype(np.int16) - prev_border_gray.astype(np.int16))
            center_diff_ratio_list.append(float(np.mean(center_diff > DIFF_THRESHOLD)))
            border_diff_ratio_list.append(float(np.mean(border_diff > DIFF_THRESHOLD)))
            center_hist_l1_list.append(float(np.abs(center_hist - prev_center_hist).sum()))
            border_hist_l1_list.append(float(np.abs(border_hist - prev_border_hist).sum()))

        prev_gray = gray
        prev_hsv = hsv
        prev_ycrcb = ycrcb
        prev_hist = hist
        prev_lap = lap
        prev_sobel = sobel
        prev_mean = cur_mean
        prev_std = cur_std
        prev_sat = cur_sat
        prev_blocks = blocks
        prev_center_gray = center_gray
        prev_border_gray = border_gray
        prev_center_hist = center_hist
        prev_border_hist = border_hist

    if len(gray_frames) == 0:
        raise ValueError(f'No frames were decoded for video_id={video_id}')

    gray_stack = np.stack(gray_frames, axis=0).astype(np.uint8)
    hist_stack = np.stack(hist_frames, axis=0).astype(np.float32)
    block3_stack = np.stack(block3_frames, axis=0).astype(np.float32)

    n_frames = gray_stack.shape[0]
    n_transitions = max(0, n_frames - 1)

    def compute_lag_features(lag):
        if n_frames <= lag:
            return {
                f'mafd_l{lag}': np.zeros(n_transitions, dtype=np.float32),
                f'diff_ratio_l{lag}': np.zeros(n_transitions, dtype=np.float32),
                f'hist_l1_l{lag}': np.zeros(n_transitions, dtype=np.float32),
                f'block_mean_l{lag}': np.zeros(n_transitions, dtype=np.float32),
            }
        diff = gray_stack[lag:].astype(np.float32) - gray_stack[:-lag].astype(np.float32)
        abs_diff = np.abs(diff)
        mafd = abs_diff.mean(axis=(1, 2)) / 255.0
        diff_ratio = (abs_diff > DIFF_THRESHOLD).mean(axis=(1, 2))
        hist_l1 = np.abs(hist_stack[lag:] - hist_stack[:-lag]).sum(axis=1)
        block_abs = np.abs(block3_stack[lag:] - block3_stack[:-lag])
        block_mean = block_abs.mean(axis=1)
        return {
            f'mafd_l{lag}': align_lag_signal(mafd, n_transitions, lag),
            f'diff_ratio_l{lag}': align_lag_signal(diff_ratio, n_transitions, lag),
            f'hist_l1_l{lag}': align_lag_signal(hist_l1, n_transitions, lag),
            f'block_mean_l{lag}': align_lag_signal(block_mean, n_transitions, lag),
        }

    signals = {
        'mae': np.asarray(mae_list, dtype=np.float32),
        'mse': np.asarray(mse_list, dtype=np.float32),
        'diff_ratio': np.asarray(diff_ratio_list, dtype=np.float32),
        'hsv': np.asarray(hsv_list, dtype=np.float32),
        'chrom': np.asarray(chrom_list, dtype=np.float32),
        'hist_l1': np.asarray(hist_list, dtype=np.float32),
        'edge': np.asarray(edge_list, dtype=np.float32),
        'sobel_diff': np.asarray(sobel_diff_list, dtype=np.float32),
        'mean_delta': np.asarray(mean_delta_list, dtype=np.float32),
        'std_delta': np.asarray(std_delta_list, dtype=np.float32),
        'sat_delta': np.asarray(sat_delta_list, dtype=np.float32),
        'y_delta': np.asarray(y_delta_list, dtype=np.float32),
        'cr_delta': np.asarray(cr_delta_list, dtype=np.float32),
        'cb_delta': np.asarray(cb_delta_list, dtype=np.float32),
        'mafd_norm': np.asarray(mafd_norm_list, dtype=np.float32),
        'center_diff_ratio': np.asarray(center_diff_ratio_list, dtype=np.float32),
        'border_diff_ratio': np.asarray(border_diff_ratio_list, dtype=np.float32),
        'center_hist_l1': np.asarray(center_hist_l1_list, dtype=np.float32),
        'border_hist_l1': np.asarray(border_hist_l1_list, dtype=np.float32),
    }
    primary_grid = 8
    for g in BLOCK_GRIDS:
        signals[f'block_mean_g{g}'] = np.asarray(block_mean_lists[g], dtype=np.float32)
        signals[f'block_max_g{g}'] = np.asarray(block_max_lists[g], dtype=np.float32)
        signals[f'block_ratio_g{g}'] = np.asarray(block_ratio_lists[g], dtype=np.float32)
    signals['block_mean'] = signals[f'block_mean_g{primary_grid}']
    signals['block_max'] = signals[f'block_max_g{primary_grid}']
    signals['block_ratio'] = signals[f'block_ratio_g{primary_grid}']
    signals.update(compute_lag_features(2))
    signals.update(compute_lag_features(3))
    signals.update(compute_heuristic_scores(signals))

    y = np.zeros(n_transitions, dtype=np.int8)
    for cut in gt.get('cut', []):
        cut = int(cut)
        if 0 <= cut < n_transitions:
            y[cut] = 1

    ignore_mask = np.zeros(n_transitions, dtype=bool)
    for key in ['trash', 'fade', 'dissolve']:
        for rng in gt.get(key, []):
            left = max(0, int(rng[0]))
            right = min(n_transitions - 1, int(rng[1]))
            if n_transitions > 0 and right >= left:
                ignore_mask[left:right + 1] = True

    return {
        'video_id': video_id,
        'video_len': int(n_frames),
        'n_transitions': int(n_transitions),
        'signals': signals,
        'y': y,
        'cuts': [int(x) for x in gt.get('cut', []) if 0 <= int(x) < n_transitions],
        'ignore_mask': ignore_mask,
        'view_cache': {},
        'candidate_cache': {},
    }


def resolve_video_path(video_item, dataset_path):
    p = Path(str(video_item))
    if p.exists():
        return p, p.stem
    dataset_path = Path(dataset_path)
    video_path = dataset_path / 'video' / f'{video_item}.mp4'
    if video_path.exists():
        return video_path, Path(video_item).stem
    raise FileNotFoundError(f'Could not resolve video: {video_item}')


def resolve_gt_path(video_id, dataset_path):
    dataset_path = Path(dataset_path)
    gt_path = dataset_path / 'gt' / f'{video_id}.json'
    if not gt_path.exists():
        raise FileNotFoundError(f'Could not resolve GT for video_id={video_id}: {gt_path}')
    return gt_path


def build_context_block(signal, prefix):
    signal = np.asarray(signal, dtype=np.float32)
    prev1 = shift_array(signal, 1)
    next1 = shift_array(signal, -1)
    mean3 = rolling_mean_1d(signal, 3)
    std3 = rolling_std_1d(signal, 3)
    peak3 = signal - 0.5 * (prev1 + next1)
    ratio3 = signal / (mean3 + 1e-6)
    robustz = robust_zscore(signal)
    roll5 = rolling_mean_1d(signal, 5)
    rollstd5 = rolling_std_1d(signal, 5)
    localpeak5 = local_peak_1d(signal, 5)
    localratio9 = local_ratio_1d(signal, 9)
    feats = [signal, prev1, next1, mean3, std3, peak3, ratio3, robustz, roll5, rollstd5, localpeak5, localratio9]
    names = [
        f'{prefix}_cur', f'{prefix}_prev1', f'{prefix}_next1',
        f'{prefix}_mean3', f'{prefix}_std3', f'{prefix}_peak3',
        f'{prefix}_ratio3', f'{prefix}_robustz', f'{prefix}_roll5',
        f'{prefix}_rollstd5', f'{prefix}_localpeak5', f'{prefix}_localratio9',
    ]
    return feats, names


def build_feature_view(entry):
    if 'exp19_exact' in entry['view_cache']:
        return entry['view_cache']['exp19_exact']
    signals = entry['signals']
    feature_arrays = []
    feature_names = []
    signal_keys = list(BASE_SIGNAL_KEYS) + list(MULTIGRID_KEYS)
    seen = set()
    signal_keys = [k for k in signal_keys if not (k in seen or seen.add(k))]
    for name in signal_keys:
        feats, names = build_context_block(signals[name], name)
        feature_arrays.extend(feats)
        feature_names.extend(names)
    X = np.stack(feature_arrays, axis=1).astype(np.float32) if feature_arrays else np.zeros((entry['n_transitions'], 0), dtype=np.float32)
    payload = {'X': X, 'columns': feature_names}
    entry['view_cache']['exp19_exact'] = payload
    return payload


def build_candidate_payload(entry):
    if 'exp19_exact' in entry['candidate_cache']:
        return entry['candidate_cache']['exp19_exact']
    score = entry['signals']['heuristic_robust']
    thr = float(np.quantile(score, EXP19_CANDIDATE_CFG['q']))
    mask = local_peak_mask(score) & (score >= thr)
    out_mask = expand_mask(mask, radius=EXP19_CANDIDATE_CFG['radius'])
    payload = {'mask': out_mask.astype(bool), 'meta': {'cand_score': score.astype(np.float32)}}
    entry['candidate_cache']['exp19_exact'] = payload
    return payload


def build_dataset(video_cache, video_ids):
    X_list, y_list = [], []
    row_video_ids, row_indices = [], []
    columns = None
    for video_id in video_ids:
        entry = video_cache[video_id]
        feat_payload = build_feature_view(entry)
        Xv = feat_payload['X']
        columns = feat_payload['columns']
        cand_payload = build_candidate_payload(entry)
        mask = cand_payload['mask']
        idx = np.where(mask)[0]
        X_sel = Xv[idx]
        col_names = list(columns)
        if cand_payload['meta']:
            meta_arrays = []
            meta_names = []
            for name, arr in cand_payload['meta'].items():
                meta_arrays.append(np.asarray(arr, dtype=np.float32)[idx][:, None])
                meta_names.append(name)
            X_sel = np.hstack([X_sel] + meta_arrays)
            col_names = col_names + meta_names
        keep = ~entry['ignore_mask'][idx]
        X_list.append(X_sel[keep])
        y_list.append(entry['y'][idx][keep].astype(np.int8))
        row_video_ids.extend([video_id] * int(np.sum(keep)))
        row_indices.extend(idx[keep].tolist())
        columns = col_names
    X = np.vstack(X_list).astype(np.float32) if X_list else np.zeros((0, 0), dtype=np.float32)
    y = np.concatenate(y_list).astype(np.int8) if y_list else np.zeros((0,), dtype=np.int8)
    return {
        'X': X,
        'y': y,
        'row_video_ids': np.asarray(row_video_ids),
        'row_indices': np.asarray(row_indices, dtype=np.int32),
        'columns': columns or [],
    }


def fit_exp19_model(video_cache, train_ids):
    dataset = build_dataset(video_cache, train_ids)
    X, y = dataset['X'], dataset['y']
    model = HistGradientBoostingClassifier(
        learning_rate=EXP19_MODEL_CFG['learning_rate'],
        max_depth=EXP19_MODEL_CFG['max_depth'],
        max_iter=EXP19_MODEL_CFG['max_iter'],
        min_samples_leaf=EXP19_MODEL_CFG['min_samples_leaf'],
        l2_regularization=EXP19_MODEL_CFG['l2_regularization'],
        random_state=EXP19_MODEL_CFG['random_state'],
    )
    model.fit(X, y)
    return {
        'feature_cfg': dict(EXP19_FEATURE_CFG),
        'candidate_cfg': dict(EXP19_CANDIDATE_CFG),
        'model_cfg': dict(EXP19_MODEL_CFG),
        'columns': dataset['columns'],
        'gate_state': None,
        'model': model,
        'threshold': EXP19_THR,
        'min_distance': EXP19_DIST,
    }


def predict_exp19_scores(state, video_id, video_cache):
    entry = video_cache[video_id]
    X_full = build_feature_view(entry)['X']
    cand_payload = build_candidate_payload(entry)
    idx = np.where(cand_payload['mask'])[0]
    X_sel = X_full[idx]
    X_sel = np.hstack([X_sel] + [np.asarray(arr, dtype=np.float32)[idx][:, None] for arr in cand_payload['meta'].values()])
    dense_scores = np.zeros(entry['n_transitions'], dtype=np.float32)
    if X_sel.shape[0] == 0:
        return dense_scores
    dense_scores[idx] = state['model'].predict_proba(X_sel)[:, 1].astype(np.float32)
    return dense_scores


def sanitize_model_for_pickle(model):
    # Keep prediction logic unchanged; remove only training-time RNG state that breaks cross-version unpickling.
    if hasattr(model, '_feature_subsample_rng'):
        model._feature_subsample_rng = None
    if hasattr(model, '_random_seed'):
        try:
            model._random_seed = int(model._random_seed)
        except Exception:
            pass
    return model


def process_video_data(video_items, dataset_path=DATASET_DIR):
    dataset_path = Path(dataset_path)
    video_cache = {}
    for video_item in video_items:
        video_path, video_id = resolve_video_path(video_item, dataset_path)
        gt_path = resolve_gt_path(video_id, dataset_path)
        frames = read_video_to_list(video_path)
        if len(frames) == 0:
            raise ValueError(f'No frames were read from {video_path}')
        gt = load_json_from_file(gt_path)
        video_cache[video_id] = extract_video_cache_from_frames(frames, video_id, gt)
        del frames
        gc.collect()
    return video_cache


def train_and_validate_models(video_cache, holdout_video_id=None):
    video_ids = list(video_cache.keys())
    if holdout_video_id is None:
        holdout_video_id = video_ids[-1]
    train_ids = [v for v in video_ids if v != holdout_video_id]
    model_state = fit_exp19_model(video_cache, train_ids)
    holdout_scores = predict_exp19_scores(model_state, holdout_video_id, video_cache)
    holdout_pred = postprocess_fixed(holdout_scores, threshold=EXP19_THR, min_distance=EXP19_DIST, peaks_only=True)
    metrics = evaluate_predictions(video_cache[holdout_video_id], holdout_pred)
    return {
        'best_model_name': 'HistGB Shallow Regularized',
        'best_f1_score': metrics['f1'],
        'model_state': model_state,
        'holdout_video_id': holdout_video_id,
        'holdout_scores': holdout_scores,
        'holdout_pred': holdout_pred,
        **metrics,
    }


# -----------------------------------------------------------------------------
# Main entry compatible with notebook style and CLI
# -----------------------------------------------------------------------------
def parse_args(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument('--dataset', default=DATASET_DIR, help='Path to dataset root')
    parser.add_argument('--holdout', default=None, help='Optional holdout video id for local evaluation')
    parser.add_argument('--model-out', default='model.pkl', help='Where to save model bundle')
    args, unknown = parser.parse_known_args(argv)
    if unknown:
        print(f'Ignoring unknown arguments: {unknown}')
    return args


def main(argv=None):
    args = parse_args(argv)
    dataset_path = Path(args.dataset)
    assert dataset_path.exists(), f'DATASET_DIR does not exist: {dataset_path}'

    if 'train_videos' in globals():
        local_train_videos = list(train_videos)
    else:
        all_videos = sorted([p.stem for p in (dataset_path / 'video').glob('*.mp4')])
        if args.holdout is not None and args.holdout in all_videos:
            local_train_videos = [v for v in all_videos if v != args.holdout] + [args.holdout]
        else:
            local_train_videos = all_videos[:-1]

    video_data = process_video_data(local_train_videos, dataset_path=dataset_path)
    default_holdout = list(video_data.keys())[-1] if len(video_data) > 0 else None
    holdout_video_id = args.holdout if args.holdout is not None else default_holdout
    results = train_and_validate_models(video_data, holdout_video_id=holdout_video_id)

    saved_state = dict(results['model_state'])
    if 'model' in saved_state:
        saved_state['model'] = sanitize_model_for_pickle(saved_state['model'])

    bundle = {
        'state': saved_state,
        'exp_name': 'HistGB Shallow Regularized',
        'threshold': EXP19_THR,
        'min_distance': EXP19_DIST,
        'resize_hw': RESIZE_HW,
        'hist_bins': HIST_BINS,
        'diff_threshold': DIFF_THRESHOLD,
        'block_grids': BLOCK_GRIDS,
        'block_mean_threshold': BLOCK_MEAN_THRESHOLD,
        'center_crop_frac': CENTER_CROP_FRAC,
        'base_signal_keys': BASE_SIGNAL_KEYS,
        'multigrid_keys': MULTIGRID_KEYS,
    }

    with open(args.model_out, 'wb') as f:
        pickle.dump(bundle, f, protocol=4)

    print('Saved model.pkl')
    print('best_model_name =', results['best_model_name'])
    print('holdout_video_id =', results['holdout_video_id'])
    print('F1 =', results['best_f1_score'])
    print('precision =', results['precision'])
    print('recall =', results['recall'])
    print('pred_count =', results['pred_count'])
    print('threshold =', EXP19_THR)
    print('min_distance =', EXP19_DIST)


if __name__ == '__main__':
    main()

Ignoring unknown arguments: ['-f', '/root/.local/share/jupyter/runtime/kernel-02a8d1aa-79b1-4f6f-8da2-905fd33da0fa.json']
Saved model.pkl
best_model_name = HistGB Shallow Regularized
holdout_video_id = 09
F1 = 0.9457364341085271
precision = 0.9104477611940298
recall = 0.9838709677419355
pred_count = 164
threshold = 0.1
min_distance = 4


In [ ]:
# GRADED CELL: scene_change_detector_ml
def scene_change_detector_ml(frames):
    import os
    import sys
    import gc
    import pickle
    import importlib
    import numpy as np
    import cv2

    RESIZE_HW = (96, 54)
    HIST_BINS = 32
    DIFF_THRESHOLD = 11
    BLOCK_GRIDS = (4, 8, 12)
    BLOCK_MEAN_THRESHOLD = 12.0
    CENTER_CROP_FRAC = 0.5
    THRESHOLD = 0.1000
    MIN_DISTANCE = 4

    BASE_SIGNAL_KEYS = [
        'mafd_norm', 'diff_ratio', 'hsv', 'chrom', 'hist_l1', 'edge',
        'mean_delta', 'std_delta', 'sat_delta',
        'block_mean', 'block_max', 'block_ratio',
        'mafd_l2', 'diff_ratio_l2', 'hist_l1_l2', 'block_mean_l2',
        'mafd_l3', 'diff_ratio_l3',
        'heuristic_fusion', 'heuristic_robust', 'heuristic_multiscale',
    ]
    MULTIGRID_KEYS = [
        'block_mean_g4', 'block_max_g4', 'block_ratio_g4',
        'block_mean_g12', 'block_max_g12', 'block_ratio_g12',
    ]

    def _install_numpy_pickle_aliases():
        try:
            import numpy as _np
            try:
                sys.modules.setdefault('numpy._core', importlib.import_module('numpy.core'))
            except Exception:
                pass
            for name in ['multiarray', 'numeric', 'numerictypes', '_multiarray_umath', 'umath']:
                try:
                    sys.modules.setdefault(f'numpy._core.{name}', importlib.import_module(f'numpy.core.{name}'))
                except Exception:
                    pass
            return _np
        except Exception:
            return None

    def _patch_numpy_random_pickle():
        try:
            import numpy.random._pickle as nrp
            import numpy.random as npr

            def _safe_ctor(bitgen_name=None):
                if isinstance(bitgen_name, type):
                    name = getattr(bitgen_name, '__name__', '')
                else:
                    name = str(bitgen_name)
                if 'MT19937' in name:
                    return npr.MT19937()
                if 'PCG64' in name:
                    return npr.PCG64()
                try:
                    return getattr(npr, name)()
                except Exception:
                    return npr.MT19937()

            nrp.__bit_generator_ctor = _safe_ctor
        except Exception:
            pass

    _install_numpy_pickle_aliases()
    _patch_numpy_random_pickle()

    def _load_bundle():
        if hasattr(scene_change_detector_ml, '_bundle_cache'):
            return scene_change_detector_ml._bundle_cache
        _install_numpy_pickle_aliases()
        _patch_numpy_random_pickle()
        try:
            with open('model.pkl', 'rb') as f:
                obj = pickle.load(f)
        except Exception:
            _install_numpy_pickle_aliases()
            _patch_numpy_random_pickle()
            with open('model.pkl', 'rb') as f:
                obj = pickle.load(f)
        scene_change_detector_ml._bundle_cache = obj
        return obj

    def shift_array(x, offset):
        x = np.asarray(x, dtype=np.float32)
        if x.size == 0 or offset == 0:
            return x.copy()
        out = np.empty_like(x)
        if offset > 0:
            out[:offset] = x[0]
            out[offset:] = x[:-offset]
        else:
            off = -offset
            out[-off:] = x[-1]
            out[:-off] = x[off:]
        return out.astype(np.float32)

    def rolling_mean_1d(x, window):
        x = np.asarray(x, dtype=np.float32)
        if x.size == 0:
            return x.copy()
        if window <= 1:
            return x.copy()
        pad = window // 2
        padded = np.pad(x, (pad, pad), mode='edge')
        kernel = np.ones(window, dtype=np.float32) / float(window)
        return np.convolve(padded, kernel, mode='valid').astype(np.float32)

    def rolling_std_1d(x, window):
        x = np.asarray(x, dtype=np.float32)
        mean = rolling_mean_1d(x, window)
        mean_sq = rolling_mean_1d(x * x, window)
        var = np.maximum(0.0, mean_sq - mean * mean)
        return np.sqrt(var).astype(np.float32)

    def local_peak_1d(x, window=5):
        return (np.asarray(x, dtype=np.float32) - rolling_mean_1d(x, window)).astype(np.float32)

    def local_ratio_1d(x, window=5):
        x = np.asarray(x, dtype=np.float32)
        return (x / (rolling_mean_1d(x, window) + 1e-6)).astype(np.float32)

    def robust_zscore(x):
        x = np.asarray(x, dtype=np.float32)
        if x.size == 0:
            return x.copy()
        med = float(np.median(x))
        mad = float(np.median(np.abs(x - med)))
        z = (x - med) / (1.4826 * mad + 1e-6)
        return np.clip(z, -10.0, 10.0).astype(np.float32)

    def percentile_scale(x, low=5.0, high=95.0):
        x = np.asarray(x, dtype=np.float32)
        if x.size == 0:
            return x.copy()
        lo, hi = np.percentile(x, [low, high])
        scaled = (x - lo) / (hi - lo + 1e-6)
        return np.clip(scaled, 0.0, 1.0).astype(np.float32)

    def local_peak_mask(scores):
        scores = np.asarray(scores, dtype=np.float32)
        if scores.size == 0:
            return np.zeros(0, dtype=bool)
        left = np.empty_like(scores)
        right = np.empty_like(scores)
        left[0] = scores[0]
        left[1:] = scores[:-1]
        right[-1] = scores[-1]
        right[:-1] = scores[1:]
        return (scores >= left) & (scores > right)

    def expand_mask(mask, radius=1):
        mask = np.asarray(mask, dtype=bool)
        if radius <= 0 or mask.size == 0:
            return mask
        idx = np.where(mask)[0]
        out = mask.copy()
        for i in idx:
            left = max(0, i - radius)
            right = min(mask.size, i + radius + 1)
            out[left:right] = True
        return out

    def greedy_distance_filter(indices, scores, min_distance):
        indices = np.asarray(indices, dtype=np.int32)
        if indices.size == 0:
            return []
        order = indices[np.argsort(scores[indices])[::-1]]
        keep = []
        for idx in order:
            ok = True
            for prev in keep:
                if abs(int(idx) - int(prev)) < int(min_distance):
                    ok = False
                    break
            if ok:
                keep.append(int(idx))
        keep = sorted(int(x) for x in keep)
        return keep

    def postprocess_fixed(scores, threshold, min_distance=1, peaks_only=True):
        scores = np.asarray(scores, dtype=np.float32)
        if scores.size == 0:
            return []
        mask = scores >= float(threshold)
        if peaks_only:
            mask = mask & local_peak_mask(scores)
        return greedy_distance_filter(np.where(mask)[0], scores, min_distance=min_distance)

    def make_hist(gray, hist_bins=32):
        hist = cv2.calcHist([gray], [0], None, [hist_bins], [0, 256]).astype(np.float32).reshape(-1)
        hist /= max(hist.sum(), 1.0)
        return hist

    def make_hist_flat(values, hist_bins=32):
        values = np.asarray(values, dtype=np.uint8).reshape(-1, 1)
        hist = cv2.calcHist([values], [0], None, [hist_bins], [0, 256]).astype(np.float32).reshape(-1)
        hist /= max(hist.sum(), 1.0)
        return hist

    def compute_block_means(gray, grid):
        h, w = gray.shape
        block_h = max(1, h // grid)
        block_w = max(1, w // grid)
        out = []
        for gy in range(grid):
            for gx in range(grid):
                y0 = gy * block_h
                y1 = h if gy == grid - 1 else (gy + 1) * block_h
                x0 = gx * block_w
                x1 = w if gx == grid - 1 else (gx + 1) * block_w
                out.append(float(gray[y0:y1, x0:x1].mean()))
        return np.asarray(out, dtype=np.float32)

    def align_lag_signal(values, n_transitions, lag):
        values = np.asarray(values, dtype=np.float32)
        out = np.zeros(n_transitions, dtype=np.float32)
        if values.size == 0 or n_transitions == 0:
            return out
        start = lag - 1
        end = min(n_transitions, start + values.size)
        out[start:end] = values[:max(0, end - start)]
        if start > 0:
            out[:start] = out[start]
        if end < n_transitions and end > 0:
            out[end:] = out[end - 1]
        return out

    def compute_heuristic_scores(base_signals):
        diff_ratio = percentile_scale(base_signals['diff_ratio'])
        hsv = percentile_scale(base_signals['hsv'])
        chrom = percentile_scale(base_signals['chrom'])
        mafd = percentile_scale(base_signals['mafd_norm'])
        block_ratio = percentile_scale(base_signals['block_ratio'])
        hist = percentile_scale(base_signals['hist_l1'])
        edge = percentile_scale(base_signals['edge'])
        sobel = percentile_scale(base_signals.get('sobel_diff', base_signals['edge']))

        heuristic_fusion = (
            0.28 * diff_ratio + 0.24 * hsv + 0.28 * chrom + 0.10 * mafd +
            0.05 * hist + 0.03 * edge + 0.02 * block_ratio
        ).astype(np.float32)
        heuristic_fusion = percentile_scale(heuristic_fusion)

        z_diff = np.clip(robust_zscore(base_signals['diff_ratio']), 0.0, None)
        z_hist = np.clip(robust_zscore(base_signals['hist_l1']), 0.0, None)
        z_block = np.clip(robust_zscore(base_signals['block_ratio']), 0.0, None)
        z_mafd = np.clip(robust_zscore(base_signals['mafd_norm']), 0.0, None)
        z_chrom = np.clip(robust_zscore(base_signals['chrom']), 0.0, None)
        z_edge = np.clip(robust_zscore(base_signals['edge']), 0.0, None)
        z_sobel = np.clip(robust_zscore(sobel), 0.0, None)

        lp_diff = np.clip(local_peak_1d(base_signals['diff_ratio'], 5), 0.0, None)
        lp_mafd = np.clip(local_peak_1d(base_signals['mafd_norm'], 5), 0.0, None)
        lp_block = np.clip(local_peak_1d(base_signals['block_ratio'], 5), 0.0, None)

        heuristic_robust = (
            0.18 * percentile_scale(z_diff) + 0.12 * percentile_scale(z_hist) +
            0.12 * percentile_scale(z_block) + 0.12 * percentile_scale(z_mafd) +
            0.10 * percentile_scale(z_chrom) + 0.08 * percentile_scale(z_edge) +
            0.06 * percentile_scale(z_sobel) + 0.08 * percentile_scale(lp_diff) +
            0.08 * percentile_scale(lp_mafd) + 0.06 * percentile_scale(lp_block)
        ).astype(np.float32)
        heuristic_robust = percentile_scale(heuristic_robust)

        heuristic_multiscale = (
            0.24 * percentile_scale(base_signals['mafd_norm']) +
            0.16 * percentile_scale(base_signals['diff_ratio']) +
            0.12 * percentile_scale(base_signals['hist_l1']) +
            0.08 * percentile_scale(base_signals['block_ratio']) +
            0.10 * percentile_scale(base_signals['mafd_l2']) +
            0.08 * percentile_scale(base_signals['diff_ratio_l2']) +
            0.05 * percentile_scale(base_signals['hist_l1_l2']) +
            0.03 * percentile_scale(base_signals['block_mean_l2']) +
            0.08 * percentile_scale(base_signals['mafd_l3']) +
            0.06 * percentile_scale(base_signals['diff_ratio_l3'])
        ).astype(np.float32)
        heuristic_multiscale = percentile_scale(heuristic_multiscale)

        heuristic_teacher = percentile_scale(0.45 * heuristic_robust + 0.35 * heuristic_multiscale + 0.20 * heuristic_fusion)

        return {
            'heuristic_fusion': heuristic_fusion,
            'heuristic_robust': heuristic_robust,
            'heuristic_multiscale': heuristic_multiscale,
            'heuristic_teacher': heuristic_teacher,
        }

    def extract_signals_from_frames(frames):
        width, height = RESIZE_HW
        gray_frames = []
        hist_frames = []
        block3_frames = []

        mae_list = []
        mse_list = []
        diff_ratio_list = []
        hsv_list = []
        chrom_list = []
        hist_list = []
        edge_list = []
        sobel_diff_list = []
        mean_delta_list = []
        std_delta_list = []
        sat_delta_list = []
        y_delta_list = []
        cr_delta_list = []
        cb_delta_list = []
        block_mean_lists = {g: [] for g in BLOCK_GRIDS}
        block_max_lists = {g: [] for g in BLOCK_GRIDS}
        block_ratio_lists = {g: [] for g in BLOCK_GRIDS}
        mafd_norm_list = []
        center_diff_ratio_list = []
        border_diff_ratio_list = []
        center_hist_l1_list = []
        border_hist_l1_list = []

        prev_gray = None
        prev_hsv = None
        prev_ycrcb = None
        prev_hist = None
        prev_lap = None
        prev_sobel = None
        prev_mean = None
        prev_std = None
        prev_sat = None
        prev_blocks = {g: None for g in BLOCK_GRIDS}
        prev_center_gray = None
        prev_border_gray = None
        prev_center_hist = None
        prev_border_hist = None

        for frame in frames:
            if frame is None:
                continue
            small = cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA)
            gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
            hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)
            ycrcb = cv2.cvtColor(small, cv2.COLOR_BGR2YCrCb)
            hist = make_hist(gray, hist_bins=HIST_BINS)
            lap = cv2.Laplacian(gray, cv2.CV_16S, ksize=3)
            sobelx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
            sobely = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
            sobel = cv2.magnitude(sobelx, sobely).astype(np.float32)

            cur_mean = float(gray.mean())
            cur_std = float(gray.std())
            cur_sat = float(hsv[:, :, 1].mean())
            block3 = cv2.resize(gray, (3, 3), interpolation=cv2.INTER_AREA).astype(np.float32).reshape(-1)
            blocks = {g: compute_block_means(gray, grid=g) for g in BLOCK_GRIDS}

            gray_frames.append(gray)
            hist_frames.append(hist)
            block3_frames.append(block3)

            h, w = gray.shape
            ch = max(1, int(round(h * CENTER_CROP_FRAC)))
            cw = max(1, int(round(w * CENTER_CROP_FRAC)))
            y0 = max(0, (h - ch) // 2)
            x0 = max(0, (w - cw) // 2)
            y1 = min(h, y0 + ch)
            x1 = min(w, x0 + cw)

            center_gray = gray[y0:y1, x0:x1]
            border_mask = np.ones_like(gray, dtype=bool)
            border_mask[y0:y1, x0:x1] = False
            border_gray = gray[border_mask]
            center_hist = make_hist(center_gray, hist_bins=HIST_BINS)
            border_hist = make_hist_flat(border_gray, hist_bins=HIST_BINS)

            if prev_gray is not None:
                diff = gray.astype(np.int16) - prev_gray.astype(np.int16)
                diff_f = diff.astype(np.float32)
                abs_diff = np.abs(diff_f)

                mae = float(abs_diff.mean())
                mse = float(np.mean(diff_f ** 2))
                mafd = float(mae / 255.0)
                diff_ratio = float(np.mean(abs_diff > DIFF_THRESHOLD))
                hsv_diff = float(np.mean(np.abs(hsv.astype(np.int16) - prev_hsv.astype(np.int16))) / 255.0)
                chrom_diff = float(np.mean(np.abs(ycrcb[:, :, 1:].astype(np.int16) - prev_ycrcb[:, :, 1:].astype(np.int16))) / 255.0)
                hist_l1 = float(np.abs(hist - prev_hist).sum())
                edge = float(np.mean(np.abs(lap.astype(np.int16) - prev_lap.astype(np.int16))) / 255.0)
                sobel_diff = float(np.mean(np.abs(sobel - prev_sobel)) / 255.0)

                y_delta = float(np.mean(np.abs(ycrcb[:, :, 0].astype(np.int16) - prev_ycrcb[:, :, 0].astype(np.int16))) / 255.0)
                cr_delta = float(np.mean(np.abs(ycrcb[:, :, 1].astype(np.int16) - prev_ycrcb[:, :, 1].astype(np.int16))) / 255.0)
                cb_delta = float(np.mean(np.abs(ycrcb[:, :, 2].astype(np.int16) - prev_ycrcb[:, :, 2].astype(np.int16))) / 255.0)

                mae_list.append(mae)
                mse_list.append(mse)
                diff_ratio_list.append(diff_ratio)
                hsv_list.append(hsv_diff)
                chrom_list.append(chrom_diff)
                hist_list.append(hist_l1)
                edge_list.append(edge)
                sobel_diff_list.append(sobel_diff)
                mean_delta_list.append(abs(cur_mean - prev_mean))
                std_delta_list.append(abs(cur_std - prev_std))
                sat_delta_list.append(abs(cur_sat - prev_sat))
                y_delta_list.append(y_delta)
                cr_delta_list.append(cr_delta)
                cb_delta_list.append(cb_delta)
                mafd_norm_list.append(mafd)

                for g in BLOCK_GRIDS:
                    block_abs_g = np.abs(blocks[g] - prev_blocks[g])
                    block_mean_lists[g].append(float(block_abs_g.mean()))
                    block_max_lists[g].append(float(block_abs_g.max()))
                    block_ratio_lists[g].append(float(np.mean(block_abs_g > BLOCK_MEAN_THRESHOLD)))

                center_diff = np.abs(center_gray.astype(np.int16) - prev_center_gray.astype(np.int16))
                border_diff = np.abs(border_gray.astype(np.int16) - prev_border_gray.astype(np.int16))
                center_diff_ratio_list.append(float(np.mean(center_diff > DIFF_THRESHOLD)))
                border_diff_ratio_list.append(float(np.mean(border_diff > DIFF_THRESHOLD)))
                center_hist_l1_list.append(float(np.abs(center_hist - prev_center_hist).sum()))
                border_hist_l1_list.append(float(np.abs(border_hist - prev_border_hist).sum()))

            prev_gray = gray
            prev_hsv = hsv
            prev_ycrcb = ycrcb
            prev_hist = hist
            prev_lap = lap
            prev_sobel = sobel
            prev_mean = cur_mean
            prev_std = cur_std
            prev_sat = cur_sat
            prev_blocks = blocks
            prev_center_gray = center_gray
            prev_border_gray = border_gray
            prev_center_hist = center_hist
            prev_border_hist = border_hist

        if len(gray_frames) == 0:
            return None

        gray_stack = np.stack(gray_frames, axis=0).astype(np.uint8)
        hist_stack = np.stack(hist_frames, axis=0).astype(np.float32)
        block3_stack = np.stack(block3_frames, axis=0).astype(np.float32)
        n_frames = gray_stack.shape[0]
        n_transitions = max(0, n_frames - 1)

        def compute_lag_features(lag):
            if n_frames <= lag:
                return {
                    f'mafd_l{lag}': np.zeros(n_transitions, dtype=np.float32),
                    f'diff_ratio_l{lag}': np.zeros(n_transitions, dtype=np.float32),
                    f'hist_l1_l{lag}': np.zeros(n_transitions, dtype=np.float32),
                    f'block_mean_l{lag}': np.zeros(n_transitions, dtype=np.float32),
                }
            diff = gray_stack[lag:].astype(np.float32) - gray_stack[:-lag].astype(np.float32)
            abs_diff = np.abs(diff)
            mafd = abs_diff.mean(axis=(1, 2)) / 255.0
            diff_ratio = (abs_diff > DIFF_THRESHOLD).mean(axis=(1, 2))
            hist_l1 = np.abs(hist_stack[lag:] - hist_stack[:-lag]).sum(axis=1)
            block_abs = np.abs(block3_stack[lag:] - block3_stack[:-lag])
            block_mean = block_abs.mean(axis=1)
            return {
                f'mafd_l{lag}': align_lag_signal(mafd, n_transitions, lag),
                f'diff_ratio_l{lag}': align_lag_signal(diff_ratio, n_transitions, lag),
                f'hist_l1_l{lag}': align_lag_signal(hist_l1, n_transitions, lag),
                f'block_mean_l{lag}': align_lag_signal(block_mean, n_transitions, lag),
            }

        signals = {
            'mae': np.asarray(mae_list, dtype=np.float32),
            'mse': np.asarray(mse_list, dtype=np.float32),
            'diff_ratio': np.asarray(diff_ratio_list, dtype=np.float32),
            'hsv': np.asarray(hsv_list, dtype=np.float32),
            'chrom': np.asarray(chrom_list, dtype=np.float32),
            'hist_l1': np.asarray(hist_list, dtype=np.float32),
            'edge': np.asarray(edge_list, dtype=np.float32),
            'sobel_diff': np.asarray(sobel_diff_list, dtype=np.float32),
            'mean_delta': np.asarray(mean_delta_list, dtype=np.float32),
            'std_delta': np.asarray(std_delta_list, dtype=np.float32),
            'sat_delta': np.asarray(sat_delta_list, dtype=np.float32),
            'y_delta': np.asarray(y_delta_list, dtype=np.float32),
            'cr_delta': np.asarray(cr_delta_list, dtype=np.float32),
            'cb_delta': np.asarray(cb_delta_list, dtype=np.float32),
            'mafd_norm': np.asarray(mafd_norm_list, dtype=np.float32),
            'center_diff_ratio': np.asarray(center_diff_ratio_list, dtype=np.float32),
            'border_diff_ratio': np.asarray(border_diff_ratio_list, dtype=np.float32),
            'center_hist_l1': np.asarray(center_hist_l1_list, dtype=np.float32),
            'border_hist_l1': np.asarray(border_hist_l1_list, dtype=np.float32),
        }
        primary_grid = 8
        for g in BLOCK_GRIDS:
            signals[f'block_mean_g{g}'] = np.asarray(block_mean_lists[g], dtype=np.float32)
            signals[f'block_max_g{g}'] = np.asarray(block_max_lists[g], dtype=np.float32)
            signals[f'block_ratio_g{g}'] = np.asarray(block_ratio_lists[g], dtype=np.float32)
        signals['block_mean'] = signals[f'block_mean_g{primary_grid}']
        signals['block_max'] = signals[f'block_max_g{primary_grid}']
        signals['block_ratio'] = signals[f'block_ratio_g{primary_grid}']
        signals.update(compute_lag_features(2))
        signals.update(compute_lag_features(3))
        signals.update(compute_heuristic_scores(signals))
        return {
            'video_len': int(n_frames),
            'n_transitions': int(n_transitions),
            'signals': signals,
            'view_cache': {},
            'candidate_cache': {},
        }

    def build_context_block(signal, prefix):
        signal = np.asarray(signal, dtype=np.float32)
        prev1 = shift_array(signal, 1)
        next1 = shift_array(signal, -1)
        mean3 = rolling_mean_1d(signal, 3)
        std3 = rolling_std_1d(signal, 3)
        peak3 = signal - 0.5 * (prev1 + next1)
        ratio3 = signal / (mean3 + 1e-6)
        robustz = robust_zscore(signal)
        roll5 = rolling_mean_1d(signal, 5)
        rollstd5 = rolling_std_1d(signal, 5)
        localpeak5 = local_peak_1d(signal, 5)
        localratio9 = local_ratio_1d(signal, 9)
        feats = [signal, prev1, next1, mean3, std3, peak3, ratio3, robustz, roll5, rollstd5, localpeak5, localratio9]
        return feats

    def build_feature_view(entry):
        if 'exp19_exact' in entry['view_cache']:
            return entry['view_cache']['exp19_exact']
        signals = entry['signals']
        feature_arrays = []
        signal_keys = list(BASE_SIGNAL_KEYS) + list(MULTIGRID_KEYS)
        seen = set()
        signal_keys = [k for k in signal_keys if not (k in seen or seen.add(k))]
        for name in signal_keys:
            feature_arrays.extend(build_context_block(signals[name], name))
        X = np.stack(feature_arrays, axis=1).astype(np.float32) if feature_arrays else np.zeros((entry['n_transitions'], 0), dtype=np.float32)
        payload = {'X': X}
        entry['view_cache']['exp19_exact'] = payload
        return payload

    def build_candidate_payload(entry):
        if 'exp19_exact' in entry['candidate_cache']:
            return entry['candidate_cache']['exp19_exact']
        score = entry['signals']['heuristic_robust']
        thr = float(np.quantile(score, 0.75))
        mask = local_peak_mask(score) & (score >= thr)
        out_mask = expand_mask(mask, radius=1)
        payload = {'mask': out_mask.astype(bool), 'meta': {'cand_score': score.astype(np.float32)}}
        entry['candidate_cache']['exp19_exact'] = payload
        return payload

    bundle = _load_bundle()
    entry = extract_signals_from_frames(frames)
    if entry is None or entry['n_transitions'] <= 0:
        return [], [], []

    X_full = build_feature_view(entry)['X']
    cand_payload = build_candidate_payload(entry)
    idx = np.where(cand_payload['mask'])[0]
    dense_scores = np.zeros(entry['n_transitions'], dtype=np.float32)
    if idx.size > 0:
        X_sel = X_full[idx]
        X_sel = np.hstack([X_sel] + [np.asarray(arr, dtype=np.float32)[idx][:, None] for arr in cand_payload['meta'].values()])
        model = bundle['state']['model'] if 'state' in bundle else bundle['model']
        dense_scores[idx] = model.predict_proba(X_sel)[:, 1].astype(np.float32)

    threshold = float(bundle.get('threshold', THRESHOLD)) if isinstance(bundle, dict) else THRESHOLD
    min_distance = int(bundle.get('min_distance', MIN_DISTANCE)) if isinstance(bundle, dict) else MIN_DISTANCE
    scene_changes = postprocess_fixed(dense_scores, threshold=threshold, min_distance=min_distance, peaks_only=True)
    metric_values = dense_scores.tolist()
    visualization = []
    gc.collect()
    return list(scene_changes), visualization, metric_values

In [ ]:
holdout_index = video_ids.index(HOLDOUT_VIDEO_ID)
holdout_f1 = run_scene_change_detector_ml_one_video(scene_change_detector_ml, str(DATASET_DIR), holdout_index)
print(f"Holdout video {HOLDOUT_VIDEO_ID}: F1 = {holdout_f1:.6f}")

Holdout video 08: F1 = 0.934132
